# Review text model (role C, task 9)

Predicts whether an active restaurant closes in the 12 months after the cutoff, from what
customers wrote in its most recent reviews before the cutoff.
Code: `src/text.py`. Labels and split: `src/labels.py`. Scoring: `src/evaluate.py`.

**Method.** Each review is turned into a 384-number vector by a small pretrained
sentence-transformer (`all-MiniLM-L6-v2`), so reviews with similar meaning get similar vectors.
A restaurant is represented by the average vector of its last 20 reviews dated on or before the
cutoff, and a logistic regression on that average predicts closure. The regularisation strength
is chosen by 5-fold cross-validation on the training year only.

Train = 2017-12-31 snapshot (2018 closures), test = 2018-12-31 snapshot (2019 closures).

The first run embeds about 1.3M reviews (about 12 minutes on an Apple-silicon GPU) and caches
them in `models/`; after that the notebook runs in a few minutes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src import config
from src.baseline import star_rating_scores
from src.evaluate import compare, evaluate
from src.labels import _end_of_day, train_test_split
from src.text import build_text_features, fit_text_model, load_text_reviews, predict
from src.data import load_restaurant_reviews

pd.set_option("display.width", 200)

In [ ]:
reviews = load_text_reviews()
dates = reviews[["business_id", "date"]]
stars = load_restaurant_reviews(columns=["business_id", "date", "stars"])
splits = {label: train_test_split(pre_cutoff_closures=label, reviews=dates)
          for label in ("closed", "exclude")}
train, test = splits["closed"]
X_train = build_text_features(train, reviews)   # last 20 reviews; embeds on first run only
X_test = build_text_features(test, reviews)
X_train.shape, X_test.shape

## Text model vs the simple rules

Each model is fitted on the training rows of its label (the `exclude` label drops restaurants
flagged closed whose last review was before the cutoff) and scored with the shared `evaluate()`.

- **Star rating**: lowest average rating first (`src/baseline.py`).
- **Recency**: longest time since the last review first.

For reference, Dian's tabular gradient boosting model (PR #8) scores 0.331 on `closed` and
0.143 on `exclude`.

In [ ]:
def days_since_last(labels):
    last = dates[dates["business_id"].isin(labels["business_id"])
                 & dates["date"].lt(_end_of_day(labels["cutoff"].iloc[0]))]
    last = last.groupby("business_id")["date"].max()
    return (_end_of_day(labels["cutoff"].iloc[0]) - last).dt.days

rows, text_scores = [], {}
for label, (tr, te) in splits.items():
    in_tr = train["business_id"].isin(tr["business_id"]).to_numpy()
    in_te = test["business_id"].isin(te["business_id"]).to_numpy()
    model = fit_text_model(X_train[in_tr], train["closed"][in_tr])
    text_scores[label] = pd.Series(predict(model, X_test[in_te]), index=test["business_id"][in_te])
    for r in [evaluate(te, star_rating_scores(te, stars), "Star rating", probabilities=False),
              evaluate(te, days_since_last(te), "Recency", probabilities=False),
              evaluate(te, text_scores[label], "Text (last 20 reviews)")]:
        rows.append({**r, "label": label})

table = pd.DataFrame(rows)
table.pivot(index="model", columns="label",
            values=["avg_precision", "roc_auc", "lift_top5"]).round(3)

## How many reviews to average

Averaging the last 10, 20 or 50 reviews gives nearly the same accuracy (in the first full
run: 0.216, 0.211 and 0.210 average precision on `closed`; 0.118, 0.127 and 0.128 on `exclude`),
so the default stays at 20: the most recent reviews describe the restaurant as it is at the cutoff.

In [ ]:
rows_n = []
for n in (10, 20, 50):
    Xa, Xb = build_text_features(train, reviews, n), build_text_features(test, reviews, n)
    m = fit_text_model(Xa, train["closed"])
    rows_n.append(evaluate(test, pd.Series(predict(m, Xb), index=test["business_id"]),
                           f"last {n} reviews"))
compare(rows_n)[["avg_precision", "roc_auc", "lift_top5", "lift_top10"]]

## Does the text add anything beyond "the restaurant went quiet"?

A restaurant that is about to close may simply get fewer reviews, and the model could be picking
that up indirectly. To check, a logistic regression on recency alone (days since the last review,
reviews in the last 3 and 6 months, total reviews; log-transformed) is compared with the same
model plus the text features. If the text only repeated recency, the second model would not do better.

In [ ]:
def recency_features(labels):
    cutoff_end = _end_of_day(labels["cutoff"].iloc[0])
    before = dates[dates["business_id"].isin(labels["business_id"]) & dates["date"].lt(cutoff_end)]
    g = before.groupby("business_id")["date"]
    f = pd.DataFrame({
        "days_since_last": (cutoff_end - g.max()).dt.days,
        "n_3m": before[before["date"].ge(cutoff_end - pd.DateOffset(months=3))].groupby("business_id").size(),
        "n_6m": before[before["date"].ge(cutoff_end - pd.DateOffset(months=6))].groupby("business_id").size(),
        "review_count": g.size(),
    }).fillna(0)
    return np.log1p(f.reindex(labels["business_id"]).to_numpy())

def logit(Xa, Xb, name):
    m = make_pipeline(StandardScaler(), LogisticRegressionCV(
        Cs=[0.001, 0.01, 0.1, 1], cv=5, scoring="average_precision", max_iter=3000,
        random_state=config.RANDOM_SEED)).fit(Xa, train["closed"])
    return evaluate(test, pd.Series(m.predict_proba(Xb)[:, 1], index=test["business_id"]), name)

R_train, R_test = recency_features(train), recency_features(test)
compare([
    logit(R_train, R_test, "Recency features"),
    logit(np.hstack([R_train, X_train]), np.hstack([R_test, X_test]), "Recency features + text"),
    logit(X_train, X_test, "Text only"),
])[["avg_precision", "roc_auc", "lift_top5", "lift_top10"]]

## What the riskiest restaurants' reviews talk about

The embedding model is not directly interpretable, so this compares the words in the recent
reviews of the 10% of test restaurants the text model rates riskiest with those of the 50% it
rates safest. Words are ranked by smoothed log-odds (how much more often they appear in the risky
group's reviews), counting each word once per review.

In [ ]:
risk = text_scores["closed"]
riskiest = risk[risk >= risk.quantile(0.9)].index
safest = risk[risk <= risk.median()].index

def recent_text(ids, n_per_group=40_000):
    cutoff_end = _end_of_day(config.CUTOFF)
    r = reviews[reviews["business_id"].isin(ids) & reviews["date"].lt(cutoff_end)]
    r = r.sort_values("date").groupby("business_id").tail(20)
    return r["text"].sample(min(n_per_group, len(r)), random_state=config.RANDOM_SEED)

risky_text, safe_text = recent_text(riskiest), recent_text(safest)
vec = CountVectorizer(stop_words="english", min_df=50, binary=True, token_pattern=r"[a-z]{3,}")
counts = vec.fit_transform(pd.concat([risky_text, safe_text]))
a = np.asarray(counts[:len(risky_text)].sum(0)).ravel() + 1
b = np.asarray(counts[len(risky_text):].sum(0)).ravel() + 1
log_odds = pd.Series(np.log(a / a.sum()) - np.log(b / b.sum()), index=vec.get_feature_names_out())
pd.DataFrame({"more common for riskiest": log_odds.nlargest(25).index,
              "more common for safest": log_odds.nsmallest(25).index})

## Findings (first full run, 2019 test set)

- **Text beats both simple rules.** Average precision 0.211 on `closed` (star rating 0.092,
  recency 0.143, closure rate 0.094) and 0.127 on `exclude` (star rating 0.059, recency 0.065).
  The riskiest 5% by text close at 3.2 times the average rate.
- **It is below the tabular model on `closed`** (0.211 vs 0.331) **but close on `exclude`**
  (0.127 vs 0.143), the label without restaurants that may already have closed before the cutoff,
  where recency carries little signal.
- **The text adds signal beyond recency**: recency features alone 0.176, plus text 0.267.
- **Much of the signal is the kind of restaurant.** Reviews of the riskiest restaurants mention
  independent, often immigrant-run cuisines (Peruvian, Turkish, Moroccan, dosa, shawarma); the
  safest mention chains (Sonic, Domino's, McDonald's, IHOP, Panda). A few words describe the
  restaurant's situation ("kinks" as in a new place working out the kinks, "reopened").
  The tabular model already has chain size and cuisine categories, so **the combined model
  (task 11) must show what text adds beyond them**; that, not text alone, is the comparison for
  the slides.